# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

---

### Dataset Source
The dataset source is provided via a Croissant schema URL:

https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json


In [ ]:
# Ensure `mlcroissant` and visualization libraries are installed
!pip install mlcroissant matplotlib seaborn

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Define dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# View metadata information as JSON
metadata = dataset.metadata.to_json()
print("Dataset Name:", metadata['name'])
print("Description:", metadata['description'])
print("Published:", metadata.get('datePublished'))

## 2. Data Overview

Review available record sets, fields, and their `@id`s.

Each record set summarizes a subset of the dataset. We inspect all available record sets by their `@id`, then examine their fields and columns (also referenced by `@id`).

In [ ]:
# List all record sets in metadata
# Each record set is referenced by its '@id'

record_sets = []
if hasattr(dataset.metadata, 'recordSet') and dataset.metadata.recordSet:
    record_sets = dataset.metadata.recordSet
elif 'recordSet' in metadata and metadata['recordSet']:
    record_sets = metadata['recordSet']

if not record_sets:
    # Try to get record sets using dataset.list_record_sets()
    record_sets = dataset.list_record_sets()

print("Available Record Sets (@id):")
for rs in record_sets:
    if isinstance(rs, dict) and '@id' in rs:
        print(f"- {rs['@id']}")
    elif isinstance(rs, str):
        print(f"- {rs}")

# Inspect fields and columns for each record set
for rs in record_sets:
    rs_id = rs['@id'] if isinstance(rs, dict) and '@id' in rs else rs
    print(f"\nRecord Set: {rs_id}")
    try:
        # Get field info (field @id, column @id)
        info = dataset.record_set(rs_id)
        print("Fields and Columns (@id):")
        for f in info['fields']:
            print(f"  Field: {f['@id']}, Name: {f.get('name')}, DataType: {f.get('dataType')}")
            if 'columns' in f:
                for c in f['columns']:
                    print(f"    Column: {c['@id']} Name: {c.get('name')}")
    except Exception as e:
        print(f"Unable to get record set info: {e}")

## 3. Data Extraction

Load data from each record set into a DataFrame for analysis.

You can specify which record set `@id` to load. Below, we extract all available record sets and store their DataFrames by their `@id`.

In [ ]:
# Get list of record set @id strings
record_set_ids = []
for rs in record_sets:
    if isinstance(rs, dict):
        rs_id = rs['@id']
    else:
        rs_id = rs
    record_set_ids.append(rs_id)

dataframes = {}
for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded Record Set '{rs_id}' with columns: {df.columns.tolist()}")
        print(df.head(), "\n")
    except Exception as e:
        print(f"Could not load records from '{rs_id}': {e}")

# Select a record set for detailed processing
example_record_set = record_set_ids[0] if record_set_ids else None
if example_record_set:
    print(f"Columns in record set {example_record_set}:")
    print(dataframes[example_record_set].columns.tolist())
    print(dataframes[example_record_set].head())

## 4. Exploratory Data Analysis (EDA)

Apply data processing steps such as filtering records based on numeric field criteria, normalizing values, and grouping.

In [ ]:
# Example: Select numeric field (by @id or column name), threshold, group field
record_set_id = example_record_set

df = dataframes[record_set_id]
# Locate a numeric column (e.g., 'age_at_diagnosis')
numeric_field_name = None
numeric_candidates = [col for col in df.columns if 'age' in col or 'height' in col or 'level' in col or 'frequency' in col]
if numeric_candidates:
    numeric_field_name = numeric_candidates[0]

# Fallback: use first numeric column
if not numeric_field_name:
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_name = col
            break

if numeric_field_name is not None:
    threshold = 10
    # Filter records with numeric_field > threshold
    filtered_df = df[df[numeric_field_name] > threshold]
    print(f"Filtered records where '{numeric_field_name}' > {threshold}:")
    print(filtered_df.head())

    # Normalize numeric field
    col_norm = numeric_field_name + '_normalized'
    filtered_df[col_norm] = (filtered_df[numeric_field_name] - filtered_df[numeric_field_name].mean()) / filtered_df[numeric_field_name].std()
    print(f"Normalized '{numeric_field_name}' for filtered records:")
    print(filtered_df[[numeric_field_name, col_norm]].head())

    # Try grouping by a categorical field (e.g., 'phenotype' or similar)
    group_field_candidates = [col for col in df.columns if 'phenotype' in col or 'zygocity' in col or 'pathogenicity' in col or 'diagnosis' in col]
    group_field = group_field_candidates[0] if group_field_candidates else (df.columns[0] if df.columns.size > 0 else None)

    if group_field and group_field != numeric_field_name:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_name].mean().to_frame()
        print(f"Grouped data by '{group_field}':")
        print(grouped_df.head())
else:
    print("No numeric fields found for EDA in this record set.")

## 5. Visualization

Visualize data distributions or relationships between fields.

We plot a histogram of the numeric field and a boxplot by group if possible.

In [ ]:
# Visualize the distribution of numeric_field in filtered_df
if numeric_field_name and not filtered_df.empty:
    plt.figure(figsize=(7,4))
    sns.histplot(filtered_df[numeric_field_name], kde=True)
    plt.title(f"Distribution of {numeric_field_name} (filtered > {threshold})")
    plt.xlabel(numeric_field_name)
    plt.show()

    if group_field and group_field != numeric_field_name and group_field in filtered_df.columns:
        plt.figure(figsize=(7,4))
        sns.boxplot(x=filtered_df[group_field], y=filtered_df[numeric_field_name])
        plt.title(f"{numeric_field_name} grouped by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_name)
        plt.show()

## 6. Conclusion

This notebook demonstrated how to load and explore the FAIR^2 dataset package using `mlcroissant`, referencing all entities by their `@id`.

- The dataset is organized in distinct record sets, each accessed via its unique `@id`.
- We've reviewed metadata, extracted data, and performed basic EDA including filtering, normalization, grouping, and simple visualizations.
- All fields and columns are handled by name and can be referenced by their `@id` when needed for reproducibility.

**Note:** The dataset is limited in sample size, so results are illustrative. For further analyses, explore each record set and field in depth as defined by its schema.